In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch.nn.functional import cosine_similarity
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms

# Phase1: Data Preparation & Preprocessing

### Data Loading

In [2]:
# csv files Reading
from dependency import *

all_indexes = pd.read_csv(all_index_path)
meta_data = pd.read_csv(img_info_path)

test_index = pd.read_csv(test_index_path)
train_index = pd.read_csv(train_index_path)

valid_index = pd.read_csv(val_index_path)

images = pd.read_csv(img_info_path)

### Metadata encoding

In [3]:
location = {name: idx for idx, name in enumerate(location_list)}

gender = {name: idx for idx, name in enumerate(sex_list)}

difficulty_level = {name: idx for idx, name in enumerate(level_of_diagnostic_difficulty_label_list)}

elevation = {name: idx for idx, name in enumerate(elevation_list)}

management = {name: idx for idx, name in enumerate(management_list)}

In [4]:
meta_data["location_id"] = meta_data["location"].map(location)
meta_data["gender_id"] = meta_data["sex"].map(gender)
meta_data["diagnostic_difficulty_level_id"] = meta_data["level_of_diagnostic_difficulty"].map(difficulty_level)
meta_data["elevation_id"] = meta_data["elevation"].map(elevation)
meta_data["management_id"] = meta_data["management"].map(management)

### (Seven points checklist + DIAG) encoding

In [5]:
# 1.
diagnosis = {}
for idx, group in enumerate(label_list):
    for label in group:
        diagnosis[label] = idx

# 2.
pigment_network = {}
for idx, group in enumerate(pigment_network_label_list):
    for label in group:
        pigment_network[label] = idx

# 3.
streaks = {}
for idx, group in enumerate(streaks_label_list):
    for label in group:
        streaks[label] = idx

# 4.
pigmentation = {}
for idx, group in enumerate(pigmentation_label_list):
    for label in group:
        pigmentation[label] = idx

# 5.
regression_structures = {}
for idx, group in enumerate(regression_structures_label_list):
    for label in group:
        regression_structures[label] = idx

# 6.
dots_and_globules = {}
for idx, group in enumerate(dots_and_globules_label_list):
    for label in group:
        dots_and_globules[label] = idx

# 7.
vascular_structures = {}
for idx, group in enumerate(vascular_structures_label_list):
    for label in group:
        vascular_structures[label] = idx

# 8.
blue_whitish_veil = {}
for idx, group in enumerate(blue_whitish_veil_label_list):
    for label in group:
        blue_whitish_veil[label] = idx

In [6]:
# 1.
meta_data["diagnosis_id"] = meta_data["diagnosis"].map(diagnosis)
# 2.
meta_data["pigment_network_id"] = meta_data["pigment_network"].map(pigment_network)
# 3.
meta_data["streaks_id"] = meta_data["streaks"].map(streaks)
# 4.
meta_data["pigmentation_id"] = meta_data["pigmentation"].map(pigmentation)
# 5.
meta_data["regression_structures_id"] = meta_data["regression_structures"].map(regression_structures)
# 6.
meta_data["dots_and_globules_id"] = meta_data["dots_and_globules"].map(dots_and_globules)
# 7.
meta_data["blue_whitish_veil_id"] = meta_data["blue_whitish_veil"].map(blue_whitish_veil)
# 8.
meta_data["vascular_structures_id"] = meta_data["vascular_structures"].map(vascular_structures)

### Chi-Squared Test

In [7]:
from statsmodels.stats.contingency_tables import Table

metadata_columns = ["diagnostic_difficulty_level_id", "elevation_id", "location_id", "management_id", "gender_id"]

for column in metadata_columns:
    contingency_table = pd.crosstab(meta_data[column], meta_data["diagnosis_id"])

    # Chi-square test
    table = Table(contingency_table)
    result = table.test_nominal_association()
    print(f"{column:32s} p-value = {result.pvalue:.6f}")

diagnostic_difficulty_level_id   p-value = 0.000003
elevation_id                     p-value = 0.000000
location_id                      p-value = 0.000000
management_id                    p-value = 0.000000
gender_id                        p-value = 0.133539


In [16]:
meta_data.head()

,case_num,diagnosis,seven_point_score,pigment_network,streaks,pigmentation,regression_structures,dots_and_globules,blue_whitish_veil,vascular_structures,...,elevation_id,management_id,diagnosis_id,pigment_network_id,streaks_id,pigmentation_id,regression_structures_id,dots_and_globules_id,blue_whitish_veil_id,vascular_structures_id
0,1,basal cell carcinoma,0,absent,absent,absent,absent,absent,absent,arborizing,...,2,0,1,0,0,0,0,0,0,1
1,2,basal cell carcinoma,1,absent,absent,absent,absent,irregular,absent,absent,...,1,0,1,0,0,0,0,2,0,0
2,3,basal cell carcinoma,1,absent,absent,absent,absent,irregular,absent,arborizing,...,1,0,1,0,0,0,0,2,0,1
3,4,basal cell carcinoma,4,absent,absent,absent,blue areas,irregular,present,within regression,...,1,0,1,0,0,0,1,2,1,1
4,5,basal cell carcinoma,1,absent,absent,diffuse irregular,absent,absent,absent,absent,...,1,0,1,0,0,2,0,0,0,0


In [21]:
meta_data['diagnosis'].value_counts()

diagnosis
clark nevus                     399
melanoma (less than 0.76 mm)    102
reed or spitz nevus              79
melanoma (in situ)               64
melanoma (0.76 to 1.5 mm)        53
seborrheic keratosis             45
basal cell carcinoma             42
dermal nevus                     33
vascular lesion                  29
blue nevus                       28
melanoma (more than 1.5 mm)      28
lentigo                          24
dermatofibroma                   20
congenital nevus                 17
melanosis                        16
combined nevus                   13
miscellaneous                     8
recurrent nevus                   6
melanoma metastasis               4
melanoma                          1
Name: count, dtype: int64

In [20]:
meta_data.columns

Index(['case_num', 'diagnosis', 'seven_point_score', 'pigment_network',
       'streaks', 'pigmentation', 'regression_structures', 'dots_and_globules',
       'blue_whitish_veil', 'vascular_structures',
       'level_of_diagnostic_difficulty', 'elevation', 'location', 'sex',
       'management', 'clinic', 'derm', 'case_id', 'notes', 'location_id',
       'gender_id', 'diagnostic_difficulty_level_id', 'elevation_id',
       'management_id', 'diagnosis_id', 'pigment_network_id', 'streaks_id',
       'pigmentation_id', 'regression_structures_id', 'dots_and_globules_id',
       'blue_whitish_veil_id', 'vascular_structures_id'],
      dtype='object')

In [27]:
meta_data['seven_point_score'].value_counts()

seven_point_score
0     311
1     166
2     165
4      96
3      91
5      81
6      50
7      28
8      14
9       7
10      2
Name: count, dtype: int64

In [28]:
meta_data['diagnosis'].value_counts()

diagnosis
clark nevus                     399
melanoma (less than 0.76 mm)    102
reed or spitz nevus              79
melanoma (in situ)               64
melanoma (0.76 to 1.5 mm)        53
seborrheic keratosis             45
basal cell carcinoma             42
dermal nevus                     33
vascular lesion                  29
blue nevus                       28
melanoma (more than 1.5 mm)      28
lentigo                          24
dermatofibroma                   20
congenital nevus                 17
melanosis                        16
combined nevus                   13
miscellaneous                     8
recurrent nevus                   6
melanoma metastasis               4
melanoma                          1
Name: count, dtype: int64

In [ ]:
meta_data[["pigment_network", "streaks", "pigmentation", "regression_structures", "dots_and_globules", "blue_whitish_veil", "vascular_structures"]].value_counts()

In [25]:
meta_data['pigment_network_id'].value_counts()

pigment_network_id
0    400
1    381
2    230
Name: count, dtype: int64

### One Hot Encoding

In [8]:
from sklearn.preprocessing import OneHotEncoder

# exclude "gender_id" because it's p-value is greater than significance value (0.05)
meta_data_columns = ["location_id", "diagnostic_difficulty_level_id", "elevation_id", "management_id"]

encoder = OneHotEncoder(sparse_output=False, dtype=int)
meta_data_onehot = encoder.fit_transform(meta_data[meta_data_columns])

In [9]:
meta_data_onehot = pd.DataFrame(meta_data_onehot, columns=encoder.get_feature_names_out(meta_data_columns), index=meta_data.index)

In [10]:
meta_data_onehot.head()

,location_id_0,location_id_1,location_id_2,location_id_3,location_id_4,location_id_5,location_id_6,location_id_7,location_id_8,diagnostic_difficulty_level_id_0,diagnostic_difficulty_level_id_1,diagnostic_difficulty_level_id_2,elevation_id_0,elevation_id_1,elevation_id_2,management_id_0,management_id_1,management_id_2
0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,1,0,0
1,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,1,0,0
2,0,0,0,0,0,1,0,0,0,0,1,0,0,1,0,1,0,0
3,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,1,0,0
4,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,1,0,0


In [11]:
# meta_data = pd.concat([meta_data, meta_data_onehot], axis=1)

### Pytorch Dataset

In [12]:
from torch.utils.data import Dataset
from PIL import Image

meta_data_columns = list(meta_data_onehot.columns)
target_columns = ["diagnosis_id", "pigment_network_id", "streaks_id", "pigmentation_id", "regression_structures_id", "dots_and_globules_id", "blue_whitish_veil_id", "vascular_structures_id"]

In [13]:
class SkinLesionDataset(Dataset):
    def __init__(self, meta_data, meta_data_onehot, split_indexes, source_dir, transform=None):
        self.meta_data = meta_data
        self.meta_data_onehot = meta_data_onehot
        self.indexes = split_indexes["indexes"].tolist()
        self.source_dir = source_dir
        self.transform = transform

    def __len__(self):
        return len(self.indexes)

    def __getitem__(self, idx):
        meta_data_index = self.indexes[idx]
        row = self.meta_data.iloc[meta_data_index]

        # image paths
        clinic_path = os.path.join(self.source_dir, row["clinic"])
        derm_path = os.path.join(self.source_dir, row["derm"])

        # image load
        clinic_image = Image.open(clinic_path).convert("RGB")
        derm_image = Image.open(derm_path).convert("RGB")

        # image transforms
        if self.transform is not None:
            clinic_image = self.transform(clinic_image)
            derm_image = self.transform(derm_image)

        # one-hot encoded metadata
        meta_data = self.meta_data_onehot.iloc[meta_data_index].values
        meta_data = torch.tensor(meta_data, dtype=torch.float32)

        # target labels
        targets = torch.tensor([row["diagnosis_id"], row["pigment_network_id"], row["streaks_id"], row["pigmentation_id"], row["regression_structures_id"], row["dots_and_globules_id"], row["blue_whitish_veil_id"], row["vascular_structures_id"]], dtype=torch.long)

        return clinic_image, derm_image, meta_data, targets

In [14]:

# random augmentations (Training transforms)
train_transform = transforms.Compose([
    transforms.Resize((224,224)), transforms.RandomHorizontalFlip(p=0.5), transforms.RandomVerticalFlip(p=0.5), transforms.ToTensor(), transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])
# mean and std are from ImageNet because we are using ImageNet pretrained ResNet-50

# Validation and test transforms
eval_transform = transforms.Compose([
    transforms.Resize((224,224)), transforms.ToTensor(), transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

In [15]:
train_dataset = SkinLesionDataset(meta_data=meta_data, meta_data_onehot=meta_data_onehot, split_indexes=train_index, source_dir=source_dir, transform=train_transform)

validation_dataset = SkinLesionDataset(meta_data=meta_data, meta_data_onehot=meta_data_onehot, split_indexes=valid_index, source_dir=source_dir, transform=eval_transform)

test_dataset = SkinLesionDataset(meta_data=meta_data, meta_data_onehot=meta_data_onehot, split_indexes=test_index, source_dir=source_dir, transform=eval_transform)

### Dataloader

In [16]:
from torch.utils.data import DataLoader

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)

validation_loader = DataLoader(validation_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

# Phase2: Model Building using pretrained ResNet50

In [17]:
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

### Image Modalities Feature extraction

In [18]:
class ResNet50FeatureExtractor(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()

        if pretrained:
            weights = ResNet50_Weights.DEFAULT
        else:
            weights = None
        self.backbone = resnet50(weights=weights)

        # final feature dimension
        self.feature_dim = self.backbone.fc.in_features
        # remove the original ImageNet classification layer
        self.backbone.fc = nn.Identity()

    def forward(self, x):
        features = self.backbone(x)
        return features

In [19]:
clinic_backbone = ResNet50FeatureExtractor(pretrained=True)
derm_backbone = ResNet50FeatureExtractor(pretrained=True)

In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using Device:", device)

clinic_backbone = clinic_backbone.to(device)
derm_backbone = derm_backbone.to(device)

Using Device: cuda


In [21]:
clinic_images, derm_images, metadata, targets = next(iter(train_loader))

clinic_images = clinic_images.to(device)
derm_images = derm_images.to(device)

# Set backbones to evaluation mode
clinic_backbone.eval()
derm_backbone.eval()

# Extract features
with torch.no_grad():
    clinic_features = clinic_backbone(clinic_images)
    derm_features = derm_backbone(derm_images)

In [22]:
# Check output dimensions
print("Clinical image batch    :", clinic_images.shape)
print("Dermoscopy image batch  :", derm_images.shape)
print("Clinical features shape :", clinic_features.shape)
print("Derm features shape     :", derm_features.shape)

Clinical image batch    : torch.Size([32, 3, 224, 224])
Dermoscopy image batch  : torch.Size([32, 3, 224, 224])
Clinical features shape : torch.Size([32, 2048])
Derm features shape     : torch.Size([32, 2048])


### Text Modality feature extractor

In [23]:
class MetaDataFeatureExtractor(nn.Module):
    def __init__(self, input_dim=18, hidden_dim=128, output_dim=128):
        super().__init__()
        self.network = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(p=0.5), nn.Linear(hidden_dim, output_dim), nn.ReLU())
        self.feature_dim = output_dim

    def forward(self, x):
        features = self.network(x)
        return features

In [24]:
meta_data_backbone = MetaDataFeatureExtractor(input_dim=18, hidden_dim=128, output_dim=128).to(device)

In [25]:
clinic_images, derm_images, metadata, targets = next(iter(train_loader))

In [26]:
metadata = metadata.to(device)

In [27]:
meta_data_backbone.eval()

with torch.no_grad():
    metadata_features = meta_data_backbone(metadata)

print("Metadata input shape :", metadata.shape)
print("Metadata feature shape :", metadata_features.shape)

Metadata input shape : torch.Size([32, 18])
Metadata feature shape : torch.Size([32, 128])


In [28]:
print("Clinical features :", clinic_features.shape)
print("Derm features     :", derm_features.shape)
print("Metadata features :", metadata_features.shape)

Clinical features : torch.Size([32, 2048])
Derm features     : torch.Size([32, 2048])
Metadata features : torch.Size([32, 128])


# Phase3: Feature Fusion

### Image Modality Feature Fusion

In [29]:
import torch.nn.functional as F

class CosCatImageFusion(nn.Module):
    def __init__(self, image_feature_dim = 2048, fused_feature_dim = 512):
        super().__init__()

        # Clinical features (2048) + Derm features (2048)
        concatenated_dim = image_feature_dim*2

        self.projection = nn.Sequential(nn.Linear(concatenated_dim, fused_feature_dim), nn.ReLU(), nn.Dropout(p=0.5))
        self.feature_dim = fused_feature_dim

    def forward(self, clinic_features, derm_features):
        concatenated_features = torch.cat((clinic_features, derm_features), dim=1)

        fused_features = self.projection(concatenated_features)

        cosine_similarity = F.cosine_similarity(clinic_features, derm_features, dim=1)

        return fused_features, cosine_similarity

In [30]:
image_fusion = CosCatImageFusion(image_feature_dim=2048, fused_feature_dim=512).to(device)

In [31]:
image_fusion.eval()
with torch.no_grad():
    image_features, cosine_similarity = image_fusion(clinic_features, derm_features)
print("Clinic features:", clinic_features.shape)
print("Derm features:", derm_features.shape)
print("Fused image features:", image_features.shape)
print("Cosine Similarity:", cosine_similarity.shape)

Clinic features: torch.Size([32, 2048])
Derm features: torch.Size([32, 2048])
Fused image features: torch.Size([32, 512])
Cosine Similarity: torch.Size([32])


### Multi-Task Prediction Heads

In [32]:
class MultiTaskHeads(nn.Module):
    def __init__(self, input_dim=512):
        super().__init__()

        self.diagnosis_head = nn.Linear(input_dim, num_label)

        self.pigment_network_head = nn.Linear(input_dim, num_pigment_network_label)

        self.streaks_head = nn.Linear(input_dim, num_streaks_label)

        self.pigmentation_head = nn.Linear(input_dim, num_pigmentation_label)

        self.regression_structures_head = nn.Linear(input_dim, num_dots_and_globules_label)

        self.dots_and_globules_head = nn.Linear(input_dim, num_dots_and_globules_label)

        self.blue_whitish_veil_head = nn.Linear(input_dim, num_blue_whitish_veil_label)

        self.vascular_structures_head = nn.Linear(input_dim, num_vascular_structures_label)

    def forward(self, fused_features):
        diagnosis = self.diagnosis_head(fused_features)
        pigment_network = self.pigment_network_head(fused_features)
        streaks = self.streaks_head(fused_features)
        pigmentation = self.pigmentation_head(fused_features)
        regression_structures = self.regression_structures_head(fused_features)
        dots_and_globules = self.dots_and_globules_head(fused_features)
        blue_whitish_veil = self.blue_whitish_veil_head(fused_features)
        vascular_structures = self.vascular_structures_head(fused_features)

        return (diagnosis, pigment_network, streaks, pigmentation, regression_structures, dots_and_globules, blue_whitish_veil, vascular_structures)

In [33]:
multi_task_heads = MultiTaskHeads(input_dim=512).to(device)

In [34]:
multi_task_heads.eval()
with torch.no_grad():
    (diagnosis, pigment_network, streaks, pigmentation, regression_structures, dots_and_globules, blue_whitish_veil, vascular_structures) = multi_task_heads(image_features)

In [35]:
print("Diagnosis                :", diagnosis.shape)
print("Pigment Network          :", pigment_network.shape)
print("Streaks                  :", streaks.shape)
print("Pigmentation             :", pigmentation.shape)
print("Regression Structures    :", regression_structures.shape)
print("Dots and Globules        :", dots_and_globules.shape)
print("Blue Whitish Veil        :", blue_whitish_veil.shape)
print("Vascular Structures      :", vascular_structures.shape)

Diagnosis                : torch.Size([32, 5])
Pigment Network          : torch.Size([32, 3])
Streaks                  : torch.Size([32, 3])
Pigmentation             : torch.Size([32, 3])
Regression Structures    : torch.Size([32, 3])
Dots and Globules        : torch.Size([32, 3])
Blue Whitish Veil        : torch.Size([32, 2])
Vascular Structures      : torch.Size([32, 3])


# Phase 4: Loss Functions and Optimizers

### Loss Function: Cross Entropy Loss

In [36]:
import torch.nn as nn

criterion = nn.CrossEntropyLoss()

def compute_total_loss(outputs, targets):
    (diagnosis, pigment_network, streaks, pigmentation, regression_structures, dots_and_globules, blue_whitish_veil, vascular_structures) = outputs

    loss_diagnosis = criterion(diagnosis, targets[:,0])
    loss_pigment_network = criterion(pigment_network, targets[:,1])
    loss_streaks = criterion(streaks, targets[:,2])
    loss_pigmentation = criterion(pigmentation, targets[:,3])
    loss_regression_structures = criterion(regression_structures, targets[:,4])
    loss_dots_and_globules = criterion(dots_and_globules, targets[:,5])
    loss_blue_whitish_veil = criterion(blue_whitish_veil, targets[:,6])
    loss_vascular_structures = criterion(vascular_structures, targets[:,7])

    total_loss = (loss_diagnosis + loss_pigment_network +  loss_streaks + loss_pigmentation +  loss_regression_structures + loss_dots_and_globules +  loss_blue_whitish_veil + loss_vascular_structures)

    return total_loss

### Optimizer

In [37]:
import torch.optim as optim
optimizer = optim.AdamW(
    list(clinic_backbone.parameters()) + list(derm_backbone.parameters()) + list(meta_data_backbone.parameters()) + list(image_fusion.parameters()) + list(multi_task_heads.parameters()),
    lr=1e-4,
    weight_decay=1e-4
)

### Learning Rate Scheduler

In [38]:
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

## Training Loop

### Training Phase

In [39]:
num_epochs=20
best_validation_loss = float("inf")

for epoch in range(num_epochs):
    clinic_backbone.train()
    derm_backbone.train()
    meta_data_backbone.train()
    image_fusion.train()
    multi_task_heads.train()

    train_loss = 0.0

    for clinic_images, derm_images, metadata, targets in train_loader:
        clinic_images= clinic_images.to(device)
        derm_images = derm_images.to(device)
        metadata = metadata.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        clinic_features = clinic_backbone(clinic_images)
        derm_features = derm_backbone(derm_images)
        metadata_features = meta_data_backbone(metadata)

        image_features, cosine_similarity = image_fusion(clinic_features, derm_features)
        outputs = multi_task_heads(image_features)

        loss = compute_total_loss(outputs, targets)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    scheduler.step()
    train_loss /= len(train_loader)

KeyboardInterrupt: 

### Validation Phase

In [40]:
print(type(loss))
print(loss)

In [41]:
clinic_backbone.eval()
derm_backbone.eval()
meta_data_backbone.eval()
image_fusion.eval()
multi_task_heads.eval()

validation_loss = 0.0
with torch.no_grad():
    for clinic_images, derm_images, meta_data, targets in validation_loader:
        clinic_images= clinic_images.to(device)
        derm_images= derm_images.to(device)
        metadata = metadata.to(device)
        targets = targets.to(device)

        clinic_features = clinic_backbone(clinic_images)
        derm_features = derm_backbone(derm_images)
        metadata_features = meta_data_backbone(metadata)

        image_features, cosine_similarity = image_fusion(clinic_features, derm_features)
        outputs = multi_task_heads(image_features)
        loss= compute_total_loss(outputs, targets)
        validation_loss += loss.item()

    validation_loss /= len(validation_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}] |"
          f"Train loss: {train_loss:.4f} | "
          f"Validation Loss: {validation_loss:.4f}")

### Saving the Best Model

In [42]:
if validation_loss < best_validation_loss:
    best_validation_loss= validation_loss

    torch.save({
        "clinic_backbone": clinic_backbone.state_dict(),
        "derm_backbone": derm_backbone.state_dict(),
        "metadata_backbone": meta_data_backbone.state_dict(),
        "image_fusion": image_fusion.state_dict(),
        "multi_task_heads": multi_task_heads.state_dict(),
        "optimizer": optimizer.state_dict(),
        "epoch": epoch,
        "validation_loss": validation_loss
    }, "best_coscatnet.pth")
    print("Best model Saved.")

In [43]:
print("Training samples  :", len(train_loader.dataset))
print("Validation samples:", len(validation_loader.dataset))

print("Training batches  :", len(train_loader))
print("Validation batches:", len(validation_loader))

print("Batch size        :", train_loader.batch_size)

Training samples  : 413
Validation samples: 203
Training batches  : 13
Validation batches: 7
Batch size        : 32


In [44]:
import time

for epoch in range(1):

    start_time = time.time()

    clinic_backbone.train()
    derm_backbone.train()
    meta_data_backbone.train()
    image_fusion.train()
    multi_task_heads.train()

    train_loss = 0.0

    for batch_idx, (clinic_images, derm_images, metadata, targets) in enumerate(train_loader):

        batch_start = time.time()

        clinic_images = clinic_images.to(device)
        derm_images = derm_images.to(device)
        metadata = metadata.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        clinic_features = clinic_backbone(clinic_images)
        derm_features = derm_backbone(derm_images)
        metadata_features = meta_data_backbone(metadata)

        image_features, cosine_similarity = image_fusion(
            clinic_features,
            derm_features
        )

        outputs = multi_task_heads(image_features)

        loss = compute_total_loss(outputs, targets)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        print(
            f"Batch {batch_idx + 1}/13 | "
            f"Loss: {loss.item():.4f} | "
            f"Time: {time.time() - batch_start:.2f}s"
        )

    print(
        f"Epoch time: {(time.time() - start_time) / 60:.2f} minutes"
    )

Batch 1/13 | Loss: 9.0029 | Time: 11.23s
Batch 2/13 | Loss: 8.7987 | Time: 7.99s
Batch 3/13 | Loss: 8.4744 | Time: 9.95s
Batch 4/13 | Loss: 8.2975 | Time: 7.27s
Batch 5/13 | Loss: 8.0095 | Time: 9.42s
Batch 6/13 | Loss: 7.8571 | Time: 7.54s
Batch 7/13 | Loss: 7.3266 | Time: 9.42s
Batch 8/13 | Loss: 7.4313 | Time: 7.38s
Batch 9/13 | Loss: 7.1214 | Time: 10.06s
Batch 10/13 | Loss: 7.0811 | Time: 6.93s
Batch 11/13 | Loss: 7.0689 | Time: 9.72s
Batch 12/13 | Loss: 6.3678 | Time: 7.35s
Batch 13/13 | Loss: 6.5221 | Time: 8.60s
Epoch time: 2.03 minutes
